# ESM-IF1 Inverse Folding

![ESM-IF1 Inverse Folding](https://proto-bio.github.io/proto-assets/images/tool/esm_if1/hero.png)

This notebook demonstrates both functions of the ESM-IF1 tool. In the first section, we use `run_esm_if1_sample` to generate new amino acid sequences conditioned on a target backbone structure. In the second section, we use `run_esm_if1_score` to evaluate how well a given sequence fits a target structure by computing average log-likelihood and perplexity.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm_if1")
display_overview("esm_if1")
display_docs_section("esm_if1", "Background")

## Available tools

In [ ]:
display_available_tools("esm_if1")

### `run_esm_if1_sample`

Given a backbone structure, ESM-IF1 generates new amino acid sequences that are predicted to fold into the target conformation. You can choose between vanilla ESM-IF1 weights or the ProteinDPO variant, which is optimized for designing thermodynamically stable proteins. Sampling temperature controls the diversity of the generated sequences, with lower temperatures producing more conservative designs that closely match the structural constraints.

In [ ]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1SampleConfig,
    ESMIF1SampleInput,
    run_esm_if1_sample,
)
from proto_tools.tools.inverse_folding.shared_data_models import (
    InverseFoldingInput,
    InverseFoldingStructureInput,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [ ]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chains_to_redesign=["A"],
)
inputs = InverseFoldingInput(inputs=[struct_input])

In [ ]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_sample")

# Configure sampling | see docs above for all fields
config = ESMIF1SampleConfig(
    num_sequences_per_structure=3,
    temperature=0.1,
    weights_variant="protein_dpo",  # or "esmif" for vanilla ESM-IF1
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
result = run_esm_if1_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_sample")

# Each design is a complete complex. ESM-IF1 redesigns only the target chain;
# all other input chains are carried over as fixed context.
for i, design_set in enumerate(result.design_sets):
    for j, design in enumerate(design_set.complexes):
        seq = design.designed_chains[0].sequence
        ll = design.metrics["log_likelihood"]
        display_seq = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Log-likelihood: {ll:.4f}")
        print(f"  Chains: {design.chain_sequence_map().keys()} | "
              f"designed: {[c.id for c in design.designed_chains]}")

In [ ]:
# An ESMIF1Design is a Complex subclass and can be passed directly to a structure predictor.
from proto_tools.tools.structure_prediction import ESMFoldInput

best_design = result.design_sets[0].complexes[0]
sp_inp = ESMFoldInput(complexes=[best_design])
print(f"ESMFoldInput accepted the design directly: {best_design.num_chains()} chain(s)")
for chain, was_designed in zip(best_design.chains, best_design.designed, strict=True):
    tag = "designed" if was_designed else "context"
    print(f"Chain {chain.id} ({tag}): {len(chain.sequence)} residues")


### `run_esm_if1_score`

ESM-IF1 can evaluate how well a given amino acid sequence fits a target backbone structure by computing average log-likelihood and perplexity. This is useful for ranking candidate sequences from other design methods, comparing wild-type to mutant sequences, or validating that a designed sequence is structurally compatible with the intended fold. Scoring uses the full complex context, so multi-chain interactions are accounted for.

In [ ]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1ScoringConfig,
    ESMIF1ScoringInput,
    ESMIF1ScoringPair,
    run_esm_if1_score,
)

In [ ]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_score")

# Score the first designed sequence against the GFP backbone. ESM-IF1 scores
# one chain at a time, so use the redesigned target chain's sequence.
designed_seq = result.design_sets[0].complexes[0].designed_chains[0].sequence

scoring_input = ESMIF1ScoringInput(
    sequence_structure_pairs=[
        ESMIF1ScoringPair(
            sequence=designed_seq,
            structure=gfp_structure,
        )
    ]
)

In [ ]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_score")

# Configure scoring | see docs above for all fields
scoring_config = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_esm_if1_score(scoring_input, scoring_config)

In [ ]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_score")

# Print scores for the designed sequence
print(f"Avg log-likelihood: {score_result.scores[0].avg_log_likelihood:.4f}")
print(f"Perplexity: {score_result.scores[0].perplexity:.4f}")

## Export Results

Sampling results can be exported to FASTA format for use in downstream sequence analysis pipelines. Scoring results can be exported to CSV or JSON format for further analysis or comparison.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="esm_if1_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'esm_if1_sequences.fasta'}")

score_result.export(name="esm_if1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'esm_if1_scores.csv'}")